# Stage C — Survival from TCGA Tumor Slides (resume-safe)

Tests whether a tumor's **appearance** (whole-slide image) predicts survival.

**Disconnect-proof:** this version saves each patient's features to your **Google Drive** as it
goes, and **skips patients already done**. So if Colab disconnects, just run everything again —
it picks up where it left off and keeps accumulating. Re-run until it reaches your target.

**Run:** Runtime > Change runtime type > **GPU** > Save, then **Run all**. Keep the tab active.

### Step 0 - Install tools

In [ ]:
!apt-get -qq install -y openslide-tools >/dev/null 2>&1
!pip -q install openslide-python scikit-survival requests >/dev/null 2>&1
import torch; print('GPU:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

### Step 1 - TCGA-BRCA survival (cBioPortal public API)

In [ ]:
import requests, pandas as pd
API='https://www.cbioportal.org/api'
rows=requests.get(f'{API}/studies/brca_tcga_pan_can_atlas_2018/clinical-data',
    params={'clinicalDataType':'PATIENT','projection':'SUMMARY','pageSize':10_000_000}).json()
cl=pd.DataFrame(rows).pivot_table(index='patientId',columns='clinicalAttributeId',values='value',aggfunc='first')
surv=pd.DataFrame(index=cl.index)
surv['time']=pd.to_numeric(cl['OS_MONTHS'],errors='coerce')
surv['event']=cl['OS_STATUS'].astype(str).str.startswith('1')
surv=surv.dropna(subset=['time']); surv=surv[surv['time']>0]
print('patients with survival:', len(surv))

### Step 2 - Diagnostic slide list (GDC), matched to survival

In [ ]:
import requests, json as _json
filt={'op':'and','content':[
 {'op':'in','content':{'field':'cases.project.project_id','value':['TCGA-BRCA']}},
 {'op':'in','content':{'field':'data_type','value':['Slide Image']}},
 {'op':'in','content':{'field':'experimental_strategy','value':['Diagnostic Slide']}}]}
params={'filters':_json.dumps(filt),'fields':'file_id,file_name,cases.submitter_id','format':'JSON','size':'20000'}
fj=requests.get('https://api.gdc.cancer.gov/files',params=params).json()
slides=[{'patient':h['cases'][0]['submitter_id'],'file_id':h['file_id']} for h in fj['data']['hits'] if h['cases'][0]['submitter_id'] in surv.index]
sl=pd.DataFrame(slides).drop_duplicates('patient')
print('slides matched to survival:', len(sl))

### Step 3a - Connect Google Drive (progress is saved here)
A popup will ask you to authorize Drive - approve it. Features are stored in a folder there,
so they survive disconnects.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
CKPT='/content/drive/MyDrive/stage_c_features'; os.makedirs(CKPT, exist_ok=True)
print('saving features to:', CKPT)
print('already saved from past runs:', len([f for f in os.listdir(CKPT) if f.endswith('.npy')]))

### Step 3b - Process slides (resume-safe; re-run after any disconnect)
Each patient: download slide -> tile -> features -> **save to Drive** -> delete slide.
Already-saved patients are skipped instantly. Raise N_PATIENTS for a stronger final result.

In [ ]:
import numpy as np, requests, openslide, torch
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms

N_PATIENTS = 60      # target; keep re-running until this many are saved
PATCHES    = 100
TILE       = 224
DOWNSAMPLE = 8
dev='cuda' if torch.cuda.is_available() else 'cpu'

w=ResNet50_Weights.IMAGENET1K_V2; net=resnet50(weights=w); net.fc=torch.nn.Identity(); net=net.eval().to(dev)
prep=transforms.Compose([transforms.ToTensor(), transforms.Normalize(w.transforms().mean,w.transforms().std)])

def is_tissue(im):
    a=np.asarray(im.convert('HSV')); return (a[:,:,1].mean()>25) and (a[:,:,2].mean()<235)

def slide_vector(path):
    s=openslide.OpenSlide(path); lvl=s.get_best_level_for_downsample(DOWNSAMPLE)
    W,H=s.level_dimensions[lvl]; ds=s.level_downsamples[lvl]
    coords=[(x,y) for y in range(0,H-TILE,TILE) for x in range(0,W-TILE,TILE)]
    np.random.RandomState(0).shuffle(coords)
    tiles=[]
    for (x,y) in coords:
        reg=s.read_region((int(x*ds),int(y*ds)),lvl,(TILE,TILE)).convert('RGB')
        if is_tissue(reg): tiles.append(prep(reg))
        if len(tiles)>=PATCHES: break
    s.close()
    if not tiles: return None
    with torch.no_grad(): f=net(torch.stack(tiles).to(dev)).cpu().numpy()
    return f.mean(0)

subset=sl.head(N_PATIENTS)
for i,r in enumerate(subset.itertuples(),1):
    fp=f'{CKPT}/{r.patient}.npy'
    if os.path.exists(fp):
        continue                         # already done in a previous run -> skip
    p=f'/content/{r.file_id}.svs'
    try:
        with requests.get(f'https://api.gdc.cancer.gov/data/{r.file_id}',stream=True,timeout=600) as resp:
            with open(p,'wb') as fh:
                for ch in resp.iter_content(1<<20): fh.write(ch)
        v=slide_vector(p)
        if v is not None: np.save(fp, v)
    except Exception as e:
        print('skip',r.patient,repr(e)[:70])
    finally:
        if os.path.exists(p): os.remove(p)
    saved=len([f for f in os.listdir(CKPT) if f.endswith('.npy')])
    print(f'{i}/{len(subset)} | saved so far: {saved}')
print('DONE THIS PASS. Saved total:', len([f for f in os.listdir(CKPT) if f.endswith('.npy')]))

### Step 4 - Survival model on the saved image features

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

feats={f[:-4]:np.load(f'{CKPT}/{f}') for f in os.listdir(CKPT) if f.endswith('.npy') and f[:-4] in surv.index}
pids=list(feats); X=np.vstack([feats[p] for p in pids])
sub=surv.loc[pids]; dur=sub['time'].to_numpy(float); evt=sub['event'].to_numpy(bool)
print(f'modeling on {len(pids)} patients, {int(evt.sum())} deaths')

if len(pids)>=25 and evt.sum()>=5:
    kf=KFold(5,shuffle=True,random_state=42); oof=np.full(len(pids),np.nan)
    for tr,te in kf.split(X):
        pipe=make_pipeline(StandardScaler(),PCA(n_components=min(10,len(tr)-1)),StandardScaler())
        Ztr=pipe.fit_transform(X[tr]); Zte=pipe.transform(X[te])
        m=CoxPHSurvivalAnalysis(alpha=1.0).fit(Ztr,Surv.from_arrays(event=evt[tr],time=dur[tr]))
        oof[te]=m.predict(Zte)
    c=concordance_index_censored(evt,dur,oof)[0]
    print(f'\nStage C image-only survival C-index (out-of-fold): {c:.3f}')
    print('(0.5 = chance. Small N -> noisy; treat as preliminary.)')
else:
    print('\nNot enough patients/deaths yet for a stable C-index.')
    print('Re-run Steps 3a+3b a few more times to accumulate more, then run this cell.')

### Step 5 - What this is, and isn't
**Done:** full weakly-supervised imaging-survival pipeline with Drive checkpointing.
**Limitations:** subset of patients, low magnification, plain averaging -> C-index is
**preliminary and noisy**. More patients = more reliable. Single cohort (TCGA).
**Next (Stage D - fusion):** combine these image vectors with Stage A gene features; test
whether genes + images beat either alone. Record your C-index in LAB_NOTEBOOK.md.